# Inter-Coder Reliability — Step 1

Sentence-level reliability for the binary task of detecting whether a sentence contains any social-group reference. Each outlet's reliability sample was coded independently by two or more annotators (anonymised as A1, A2, A3, ...). Reliability is reported as Krippendorff's α (Krippendorff 2004), computed across all annotators per outlet.

In [ ]:
import ast
import json
from itertools import combinations
from pathlib import Path

import pandas as pd
import krippendorff

In [ ]:
BASE_DIR = Path('./reliability_step_1')
OUTPUT_DIR = BASE_DIR / 'results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTLETS = [
    # (display_name, country, folder)
    ('Le Figaro',              'France',  'figaro'),
    ('Le Monde',               'France',  'monde'),
    ('Le Monde diplomatique',  'France',  'mondediplo'),
    ('Le Parisien',            'France',  'parisien'),
    ('Médiapart',              'France',  'mediapart'),
    ('Libération',             'France',  'libe'),
    ('Bild',                   'Germany', 'bild'),
    ('Frankfurter Allgemeine', 'Germany', 'faz'),
    ('Spiegel',                'Germany', 'spiegel'),
    ('Süddeutsche Zeitung',    'Germany', 'sz'),
    ('Die Welt',               'Germany', 'welt'),
    ('Die Zeit',               'Germany', 'zeit'),
]

## Data loading

Each outlet folder contains one Doccano JSONL file per annotator. Files within an outlet are row-aligned by sentence position. For each sentence, the annotator's spans are reduced to a binary indicator (1 if any span was marked, 0 otherwise).

In [ ]:
EMPTY_TOKENS = {0, 0.0, '0', '', '[]', '0.0', 'nan', None}


def has_annotation(label_field) -> int:
    if isinstance(label_field, list):
        return 1 if any(isinstance(s, list) and len(s) >= 2 for s in label_field) else 0
    try:
        if pd.isna(label_field):
            return 0
    except (TypeError, ValueError):
        pass
    if label_field in EMPTY_TOKENS:
        return 0
    s = str(label_field).strip()
    if s in EMPTY_TOKENS:
        return 0
    try:
        parsed = ast.literal_eval(s)
    except (ValueError, SyntaxError):
        return 0
    return 1 if isinstance(parsed, list) and any(
        isinstance(x, list) and len(x) >= 2 for x in parsed
    ) else 0


def load_outlet(folder_name: str):
    """Load all annotator JSONL files from an outlet folder.

    Returns (n_annotators, binary_matrix). Annotator identities are not retained;
    files are loaded in alphabetical order and treated as anonymous rows.
    """
    folder = BASE_DIR / folder_name
    files = sorted(folder.glob('*.jsonl'))
    if not files:
        raise FileNotFoundError(f'No JSONL files in {folder}')

    rows_by_file = []
    for p in files:
        with open(p, 'r', encoding='utf-8') as f:
            rows_by_file.append([json.loads(line) for line in f if line.strip()])

    n = min(len(rs) for rs in rows_by_file)
    binary_matrix = [[has_annotation(rs[i].get('label', [])) for i in range(n)]
                     for rs in rows_by_file]
    return len(files), binary_matrix

## Per-outlet α calculation

In [ ]:
results = []
loaded = {}

for display, country, folder in OUTLETS:
    n_annotators, binary_matrix = load_outlet(folder)
    alpha = round(
        float(krippendorff.alpha(reliability_data=binary_matrix, level_of_measurement='nominal')),
        4,
    )
    results.append({
        'Outlet':         display,
        'Country':        country,
        'Krippendorff α': alpha,
        '# Sentences':    len(binary_matrix[0]),
        '# Annotators':   n_annotators,
    })
    loaded[display] = (n_annotators, binary_matrix)

df_outlets = pd.DataFrame(results)
df_outlets.to_csv(OUTPUT_DIR / 'icr_step1_by_outlet.csv', index=False)
df_outlets

## Printout

In [ ]:
print(f'{"Newspaper":<28} {"α":>6}  {"N":>6}  {"coders":>6}')
print('-' * 56)
for country_name, code in [('French Newspapers', 'France'), ('German Newspapers', 'Germany')]:
    print(country_name)
    sub = df_outlets[df_outlets['Country'] == code]
    for _, r in sub.iterrows():
        print(f'  {r["Outlet"]:<26} {r["Krippendorff α"]:>6.2f}  '
              f'{r["# Sentences"]:>6,}  {r["# Annotators"]:>6}')

## Pairwise breakdown (outlets with more than two annotators)

In [ ]:
pair_rows = []
for display, (n_ann, matrix) in loaded.items():
    if n_ann < 3:
        continue
    country = next(c for d, c, _ in OUTLETS if d == display)
    for i, j in combinations(range(n_ann), 2):
        pair_alpha = round(
            float(krippendorff.alpha(
                reliability_data=[matrix[i], matrix[j]],
                level_of_measurement='nominal',
            )),
            4,
        )
        pair_rows.append({
            'Outlet':         display,
            'Country':        country,
            'Pair':           f'A{i+1} + A{j+1}',
            'Krippendorff α': pair_alpha,
            'N':              len(matrix[0]),
        })

df_pairs = pd.DataFrame(pair_rows)
df_pairs.to_csv(OUTPUT_DIR / 'icr_step1_by_annotator_pair.csv', index=False)
df_pairs.set_index(['Outlet', 'Pair'])